# Data Engineering — Moduł 5: dbt — testy, dokumentacja i wzorce zaawansowane

Moduł 4 nauczył nas BUDOWAĆ modele. Ten moduł uczy, jak sprawić, żeby te modele były
GOTOWE NA PRODUKCJĘ: automatyczne testy jakości danych, dokumentacja, reużywalne
fragmenty kodu (macros), małe tabele referencyjne (seeds) oraz modele przyrostowe
(incremental) dla dużych tabel faktów.

## Spis treści
1. [`dbt test` — testy generyczne](#sec1)
2. [Testy singularne — customowy SQL](#sec2)
3. [`dbt docs` — dokumentacja projektu](#sec3)
4. [Macros — reużywalne fragmenty SQL](#sec4)
5. [Seeds — małe tabele referencyjne](#sec5)
6. [Modele przyrostowe (incremental)](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Generujemy nasz znany zestaw danych sklepu internetowego i ładujemy go do **DuckDB** (lekka, plikowa baza analityczna) jako schemat `raw` -- symulując surowe dane, które w prawdziwym projekcie załadowałby zewnętrzny proces ingestii (Moduł 1). Budujemy też szkielet projektu dbt (`dbt_project.yml`, `profiles.yml`, `sources.yml`) wskazujący na te dane.

In [ ]:
import os
import subprocess
import duckdb
import numpy as np
import pandas as pd

# --- 1. Generujemy surowe dane (ten sam generator co w kursie Databricks/PySpark) ---
rng = np.random.default_rng(7)

n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=[0.35, 0.45, 0.20]),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}
kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2) for k in kategorie_produktow],
})

n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values
prawdop_anulowania = np.where(segment_zamowien == "Nowy", 0.22, np.where(segment_zamowien == "Standardowy", 0.10, 0.04))
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(anulowane == 1, "Anulowane", rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]))
data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)
zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]
for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id, "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id), "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]), "rabat": float(rabat),
        })
        pozycja_id += 1
pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- 2. Ladujemy surowe dane do DuckDB, schemat "raw" (jakby zaladowal je zewnetrzny proces ingestii) ---
DBT_DIR = os.path.abspath("dbt_kurs")
for _poddir in ("models", "tests", "macros", "seeds", "snapshots"):
    os.makedirs(f"{DBT_DIR}/{_poddir}", exist_ok=True)
DUCKDB_PATH = f"{DBT_DIR}/kurs_de.duckdb"

con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.klienci AS SELECT * FROM klienci_pd")
con.execute("CREATE OR REPLACE TABLE raw.produkty AS SELECT * FROM produkty_pd")
con.execute("CREATE OR REPLACE TABLE raw.zamowienia AS SELECT * FROM zamowienia_pd")
con.execute("CREATE OR REPLACE TABLE raw.pozycje_zamowien AS SELECT * FROM pozycje_zamowien_pd")
con.close()

# --- 3. Szkielet projektu dbt ---
with open(f"{DBT_DIR}/dbt_project.yml", "w") as f:
    f.write(
        "name: kurs_de\n"
        "version: '1.0'\n"
        "profile: kurs_de\n"
        "model-paths: [\"models\"]\n"
        "seed-paths: [\"seeds\"]\n"
        "snapshot-paths: [\"snapshots\"]\n"
    )

with open(f"{DBT_DIR}/profiles.yml", "w") as f:
    f.write(
        "kurs_de:\n"
        "  target: dev\n"
        "  outputs:\n"
        "    dev:\n"
        "      type: duckdb\n"
        f"      path: {DUCKDB_PATH}\n"
    )

with open(f"{DBT_DIR}/models/sources.yml", "w") as f:
    f.write(
        "version: 2\n"
        "sources:\n"
        "  - name: raw_dane\n"
        "    schema: raw\n"
        "    tables:\n"
        "      - name: klienci\n"
        "      - name: produkty\n"
        "      - name: zamowienia\n"
        "      - name: pozycje_zamowien\n"
    )

env = os.environ.copy()
env["DBT_PROFILES_DIR"] = DBT_DIR


def dbt(*args, check=True):
    """Uruchamia komende `dbt ...` w naszym projekcie i wypisuje wynik."""
    wynik = subprocess.run(
        ["dbt", *args, "--project-dir", DBT_DIR],
        env=env, capture_output=True, text=True,
    )
    print(wynik.stdout[-4000:])
    if wynik.returncode != 0:
        print(wynik.stderr[-2000:])
        if check:
            raise RuntimeError(f"dbt {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


print(f"Zaladowano surowe dane do {DUCKDB_PATH} (schemat raw).")
print(f"Projekt dbt: {DBT_DIR}")


In [ ]:
model_kod = '''
select
    zamowienie_id,
    klient_id,
    data_zamowienia,
    status
from {{ source('raw_dane', 'zamowienia') }}
'''
with open(f"{DBT_DIR}/models/stg_zamowienia.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_zamowienia")


<a id="sec1"></a>
## 1. `dbt test` — testy generyczne

**Testy generyczne** to gotowe reguły jakości danych, które przypisujesz do kolumn w
pliku `schema.yml` -- bez pisania SQL-a od zera. Cztery wbudowane testy pokrywają
większość codziennych potrzeb:

- **`not_null`** -- kolumna nie może zawierać braków.
- **`unique`** -- wartości w kolumnie muszą być unikalne (klucz główny).
- **`accepted_values`** -- wartość musi być z zamkniętej listy dozwolonych opcji.
- **`relationships`** -- każda wartość w tej kolumnie MUSI istnieć jako wartość klucza
  w INNYM modelu (integralność referencyjna, jak `FOREIGN KEY` w SQL z Modułu 8 kursu
  SQL).

In [ ]:
schema_yml = '''
version: 2

models:
  - name: stg_zamowienia
    columns:
      - name: zamowienie_id
        tests:
          - unique
          - not_null
      - name: klient_id
        tests:
          - not_null
          - relationships:
              to: source('raw_dane', 'klienci')
              field: klient_id
      - name: status
        tests:
          - accepted_values:
              arguments:
                values: ["Zlozone", "Wyslane", "Dostarczone", "Anulowane"]
'''
with open(f"{DBT_DIR}/models/schema.yml", "w") as f:
    f.write(schema_yml)

dbt("test", "--select", "stg_zamowienia")


> ⚠️ **Testy odpalają się na WYNIKU modelu, nie na kodzie SQL**
>
> `dbt test` NIE sprawdza, czy Twój SQL jest "poprawny składniowo" (to robi już `dbt run`) -- sprawdza, czy DANE w zbudowanym modelu spełniają regułę. Test może więc przejść dzisiaj i nagle zacząć failować jutro, gdy do źródła trafią nieoczekiwane dane (np. nowy, nieznany wcześniej status zamówienia) -- i o to właśnie chodzi: `dbt test` to CIĄGŁY monitoring jakości, uruchamiany przy każdym `dbt build`.

<a id="sec2"></a>
## 2. Testy singularne — customowy SQL

Gdy cztery gotowe testy generyczne nie wystarczają (np. reguła biznesowa specyficzna dla
Twoich danych), piszesz **test singularny** -- zwykły plik `.sql` w folderze `tests/`,
którego zapytanie ma zwrócić WIERSZE ŁAMIĄCE regułę. Test PRZECHODZI, gdy zapytanie
zwraca ZERO wierszy.

In [ ]:
model_kod = '''
select * from {{ source('raw_dane', 'pozycje_zamowien') }}
'''
with open(f"{DBT_DIR}/models/stg_pozycje.sql", "w") as f:
    f.write(model_kod)

test_kod = '''
-- Test PRZECHODZI, jesli rabat jest zawsze w rozsadnym zakresie [0, 1]
select *
from {{ ref('stg_pozycje') }}
where rabat < 0 or rabat > 1
'''
with open(f"{DBT_DIR}/tests/rabat_w_zakresie.sql", "w") as f:
    f.write(test_kod)

dbt("run", "--select", "stg_pozycje")
dbt("test", "--select", "rabat_w_zakresie")


> ⚡ **Nazwa pliku testu = nazwa testu w logach**
>
> Podobnie jak w modelach, nazwa pliku (`rabat_w_zakresie.sql`) staje się nazwą testu widoczną w logach (`PASS rabat_w_zakresie`) -- warto nadawać jej opisową nazwę, żeby od razu było wiadomo, JAKĄ regułę biznesową sprawdza, bez otwierania pliku.

<a id="sec3"></a>
## 3. `dbt docs` — dokumentacja projektu

`dbt docs generate` skanuje CAŁY projekt (modele, kolumny, testy, zależności z `ref()`/
`source()`) i buduje z tego **automatyczną dokumentację** -- interaktywną stronę z
opisem każdego modelu, jego kolumn oraz GRAFEM ZALEŻNOŚCI (dokładnie tym, który dbt
buduje wewnętrznie z `ref()`). W prawdziwym projekcie `dbt docs serve` otwiera tę stronę w
przeglądarce (u nas, w notebooku, pomijamy `serve` -- wystarczy `generate`, żeby
zobaczyć, co dokładnie zostało wygenerowane).

In [ ]:
dbt("docs", "generate")

import os
print("Wygenerowane pliki dokumentacji:")
for nazwa in ["catalog.json", "manifest.json"]:
    sciezka = f"{DBT_DIR}/target/{nazwa}"
    print(f"  {nazwa}: {'OK, ' + str(os.path.getsize(sciezka)) + ' bajtow' if os.path.exists(sciezka) else 'BRAK'}")


> 🏭 **`manifest.json` to nie tylko dokumentacja**
>
> `manifest.json` (pełny opis grafu projektu) jest też odczytywany przez INNE narzędzia -- np. Airflow może go sparsować, żeby automatycznie zbudować DAG odzwierciedlający dokładną kolejność modeli dbt (zamiast ręcznie wywoływać `dbt run` jako jeden wielki, nieprzezroczysty krok w Airflow).

<a id="sec4"></a>
## 4. Macros — reużywalne fragmenty SQL

**Macro** (makro) to fragment kodu Jinja+SQL, który możesz wywołać w wielu modelach --
odpowiednik FUNKCJI w Pythonie. Zapisujesz go RAZ w folderze `macros/`, a potem
wywołujesz przez `{{ nazwa_makra(argumenty) }}` w dowolnym modelu.

In [ ]:
import os
os.makedirs(f"{DBT_DIR}/macros", exist_ok=True)

macro_kod = '''
{% macro dodaj_vat(kolumna_ceny) %}
    round({{ kolumna_ceny }} * 1.23, 2)
{% endmacro %}
'''
with open(f"{DBT_DIR}/macros/dodaj_vat.sql", "w") as f:
    f.write(macro_kod)

model_kod = '''
select
    produkt_id,
    kategoria,
    cena,
    {{ dodaj_vat("cena") }} as cena_z_vat
from {{ source('raw_dane', 'produkty') }}
'''
with open(f"{DBT_DIR}/models/stg_produkty.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_produkty")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.stg_produkty LIMIT 5").fetchdf())
con.close()


> ⚡ **Macros to Jinja, nie "prawdziwy" SQL**
>
> Wszystko w `{{ }}` i `{% %}` to składnia szablonów **Jinja** (ten sam silnik szablonów, którego używa m.in. Flask w Pythonie) -- dbt PRZETWARZA cały plik przez Jinja PRZED wysłaniem go do bazy danych. Stąd `dbt compile` (Moduł 4) jest tak przydatny: pokazuje dokładnie, jaki SQL wyszedł z tego "programowania" w Jinja.

<a id="sec5"></a>
## 5. Seeds — małe tabele referencyjne

**Seed** to mały plik `.csv` w folderze `seeds/`, który dbt ładuje do hurtowni jako
zwykłą tabelę komendą `dbt seed`. Idealne dla małych, RZADKO ZMIENIAJĄCYCH SIĘ danych
referencyjnych, które trzymasz w Git razem z kodem (a nie w zewnętrznym systemie
źródłowym): mapowania kodów krajów, kursy walut, listy świąt, przypisania
miast do regionów.

In [ ]:
import os
os.makedirs(f"{DBT_DIR}/seeds", exist_ok=True)

seed_csv = (
    "miasto,region\n"
    "Warszawa,Mazowieckie\n"
    "Krakow,Malopolskie\n"
    "Gdansk,Pomorskie\n"
    "Wroclaw,Dolnoslaskie\n"
    "Poznan,Wielkopolskie\n"
)
with open(f"{DBT_DIR}/seeds/mapa_regionow.csv", "w") as f:
    f.write(seed_csv)

dbt("seed")


In [ ]:
model_kod = '''
select
    klient_id,
    miasto,
    segment
from {{ source('raw_dane', 'klienci') }}
'''
with open(f"{DBT_DIR}/models/stg_klienci.sql", "w") as f:
    f.write(model_kod)

# Seed jest tabela jak kazda inna -- odwolujemy sie do niej przez ref(), NIE source()!
model_kod = '''
select
    k.klient_id,
    k.miasto,
    r.region,
    k.segment
from {{ ref('stg_klienci') }} k
left join {{ ref('mapa_regionow') }} r on k.miasto = r.miasto
'''
with open(f"{DBT_DIR}/models/klienci_z_regionem.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_klienci klienci_z_regionem")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.klienci_z_regionem LIMIT 5").fetchdf())
con.close()


> ⚠️ **Seeds to NIE miejsce na duże albo często zmieniające się dane**
>
> Seed ładuje się w całości przy KAŻDYM `dbt seed` -- świetne dla tabeli z 5-50 wierszami, fatalne dla czegokolwiek większego (to zadanie dla źródeł/`source()`, ładowanych przez prawdziwy proces ingestii z Modułu 1, nie dla plików trzymanych w Git).

<a id="sec6"></a>
## 6. Modele przyrostowe (incremental)

Materializacja `incremental` (wspomniana w Module 4) unika przeliczania CAŁEJ tabeli przy
każdym uruchomieniu -- kluczowe dla dużych tabel faktów, gdzie przeliczanie miliardów
wierszy codziennie byłoby zbyt kosztowne. Makro **`is_incremental()`** zwraca `True`
TYLKO wtedy, gdy tabela już istnieje i uruchamiasz zwykły `dbt run` (nie pierwszy raz, nie
`--full-refresh`) -- pozwala to napisać JEDNO zapytanie z dwoma trybami działania:
pełnym (za pierwszym razem) i przyrostowym (za kolejnymi).

In [ ]:
# Symulujemy "dzien 1": w zrodle jest tylko czesc zamowien (starsze niz 2024-06-01)
con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE OR REPLACE TABLE raw.zamowienia AS SELECT * FROM raw.zamowienia WHERE data_zamowienia < DATE '2024-06-01'")
liczba_dzien1 = con.execute("SELECT COUNT(*) FROM raw.zamowienia").fetchone()[0]
con.close()
print(f"Dzien 1 -- w zrodle jest {liczba_dzien1} zamowien.")

model_kod = '''
{{ config(materialized="incremental", unique_key="zamowienie_id") }}

select
    zamowienie_id,
    klient_id,
    data_zamowienia,
    status
from {{ source('raw_dane', 'zamowienia') }}

{% if is_incremental() %}
where data_zamowienia > (select coalesce(max(data_zamowienia), '1900-01-01') from {{ this }})
{% endif %}
'''
with open(f"{DBT_DIR}/models/inc_zamowienia.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "inc_zamowienia")

con = duckdb.connect(DUCKDB_PATH)
print(f"Wierszy w inc_zamowienia po 1. uruchomieniu: {con.execute('SELECT COUNT(*) FROM main.inc_zamowienia').fetchone()[0]}")
con.close()


In [ ]:
# Symulujemy "dzien 2": do zrodla docieraja NOWE zamowienia (caly pozostaly zakres dat)
con = duckdb.connect(DUCKDB_PATH)
con.execute(
    "CREATE OR REPLACE TABLE raw.zamowienia AS "
    "SELECT zamowienie_id, klient_id, data_zamowienia, status FROM zamowienia_pd"
)
liczba_dzien2 = con.execute("SELECT COUNT(*) FROM raw.zamowienia").fetchone()[0]
con.close()
print(f"Dzien 2 -- w zrodle jest juz {liczba_dzien2} zamowien (caly zbior).")

# ZWYKLE 'dbt run' -- BEZ --full-refresh -- przetworzy TYLKO nowe wiersze
dbt("run", "--select", "inc_zamowienia")

con = duckdb.connect(DUCKDB_PATH)
print(f"Wierszy w inc_zamowienia po 2. uruchomieniu: {con.execute('SELECT COUNT(*) FROM main.inc_zamowienia').fetchone()[0]}")
con.close()


> ⚡ **`--full-refresh` gdy coś pójdzie nie tak**
>
> Jeśli logika przyrostowa się "rozjedzie" z rzeczywistością (np. zmieniłeś definicję modelu i stare wiersze mają zły kształt), `dbt run --full-refresh` każe dbt ZBURZYĆ i zbudować tabelę OD ZERA, ignorując `is_incremental()` -- bezpieczny "reset", którego używa się okazjonalnie, nie przy każdym uruchomieniu.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- testy generyczne (`not_null`, `unique`, `accepted_values`, `relationships`) w
  `schema.yml`,
- testy singularne -- customowy SQL zwracający wiersze łamiące regułę,
- `dbt docs generate` -- automatyczną dokumentację i graf zależności,
- macros -- reużywalne fragmenty Jinja+SQL,
- seeds -- małe tabele referencyjne ładowane z plików CSV w repo,
- modele przyrostowe (`incremental`, `is_incremental()`, `--full-refresh`).

### 📝 Ćwiczenie 1: Test relationships na pozycjach zamówień

Dodaj do `schema.yml` test `relationships` sprawdzający, że każdy `produkt_id` w `stg_pozycje` istnieje w `source('raw_dane', 'produkty')`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
schema_yml = '''
version: 2

models:
  - name: stg_pozycje
    columns:
      - name: produkt_id
        tests:
          - relationships:
              to: source('raw_dane', 'produkty')
              field: produkt_id
'''
with open(f"{DBT_DIR}/models/schema_pozycje.yml", "w") as f:
    f.write(schema_yml)

dbt("test", "--select", "stg_pozycje")
```

</details>

### 📝 Ćwiczenie 2: Test singularny na cenach

Napisz test singularny `cena_dodatnia.sql`, który PRZECHODZI, jeśli w `stg_produkty` nie ma ani jednego wiersza z `cena <= 0`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
test_kod = '''
select * from {{ ref('stg_produkty') }} where cena <= 0
'''
with open(f"{DBT_DIR}/tests/cena_dodatnia.sql", "w") as f:
    f.write(test_kod)

dbt("test", "--select", "cena_dodatnia")
```

</details>

### 📝 Ćwiczenie 3: Makro liczące marżę

Napisz makro `marza(cena, koszt)`, które zwraca `round((cena - koszt) / cena * 100, 1)` (marża procentowa), i użyj go w nowym modelu `stg_produkty_marza`, zakładając koszt jako 60% ceny (`cena * 0.6`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
macro_kod = '''
{% macro marza(cena, koszt) %}
    round(({{ cena }} - {{ koszt }}) / {{ cena }} * 100, 1)
{% endmacro %}
'''
with open(f"{DBT_DIR}/macros/marza.sql", "w") as f:
    f.write(macro_kod)

model_kod = '''
select
    produkt_id,
    cena,
    cena * 0.6 as koszt_szacowany,
    {{ marza("cena", "cena * 0.6") }} as marza_procent
from {{ source('raw_dane', 'produkty') }}
'''
with open(f"{DBT_DIR}/models/stg_produkty_marza.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_produkty_marza")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.stg_produkty_marza LIMIT 5").fetchdf())
con.close()
```

</details>

> 🔥 **Wyzwanie: seed + incremental + testy razem**
>
> Zbuduj: (1) seed `progi_segmentow.csv` z kolumnami `segment,min_zamowien` (`Nowy,0`, `Standardowy,3`, `Premium,10`), (2) model incremental `inc_klienci_aktywnosc.sql` liczący liczbę zamówień per klient (z `source('raw_dane','zamowienia')`, `unique_key='klient_id'`), (3) test `relationships` sprawdzający, że każdy `segment` klienta w `stg_klienci` istnieje w seedzie `progi_segmentow`. Uruchom `dbt seed`, `dbt run`, `dbt test` w tej kolejności.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
seed_csv = (
    "segment,min_zamowien\n"
    "Nowy,0\n"
    "Standardowy,3\n"
    "Premium,10\n"
)
with open(f"{DBT_DIR}/seeds/progi_segmentow.csv", "w") as f:
    f.write(seed_csv)

model_kod = '''
{{ config(materialized="incremental", unique_key="klient_id") }}

select
    klient_id,
    count(*) as liczba_zamowien
from {{ source('raw_dane', 'zamowienia') }}
group by klient_id
'''
with open(f"{DBT_DIR}/models/inc_klienci_aktywnosc.sql", "w") as f:
    f.write(model_kod)

schema_yml = '''
version: 2

models:
  - name: stg_klienci
    columns:
      - name: segment
        tests:
          - relationships:
              to: ref('progi_segmentow')
              field: segment
'''
with open(f"{DBT_DIR}/models/schema_segmenty.yml", "w") as f:
    f.write(schema_yml)

dbt("seed")
dbt("run")
dbt("test")
```

Zwroc uwage: model incremental grupuje PO kliencie (agregacja), wiec przy kazdym uruchomieniu przelicza SUME OD NOWA dla kazdego klienta z unique_key -- to typowy wzorzec 'incremental po encji', inny niz nasz przyklad z sekcji 6 'incremental po dacie'.

</details>

## Co dalej?

W **Module 6** przechodzimy od pojedynczych modeli do CAŁEJ ARCHITEKTURY hurtowni:
modelowanie wymiarowe (Kimball), fakty i wymiary, schemat gwiazdy -- czyli JAK
zorganizować dziesiątki modeli dbt w spójną, analitycznie użyteczną strukturę.